# Práctica 7 - Train, Validation y Test
**Sistemas de Aprendizaje Automático · Sesión 7**   
**Archivo de datos:** `clientes_ecommerce.csv`

### Contexto
La empresa de comercio electrónico quiere predecir si un cliente comprará durante el próximo mes. En esta práctica separarás correctamente los datos, evitarás fugas de información y entrenarás un primer clasificador **sin utilizar el conjunto de test para evaluarlo**.

**Instrucciones:** completa las celdas con `TODO`, ejecuta el notebook en orden y contesta en Markdown las preguntas. No cambies la semilla `random_state=42`. El archivo CSV debe estar en la misma carpeta que el notebook.

**Objetos finales:** `X_train`, `X_val`, `X_test`, `y_train`, `y_val`, `y_test`, `preprocesador`, `modelo`, `predicciones_train`, `predicciones_val`, `accuracy_train` y `accuracy_val`.

## Parte 1 - Preparación del dataset
### Ejercicio 1. Cargar y explorar
Carga los datos, muestra las primeras cinco filas, las dimensiones y los tipos de datos. Comprueba que el CSV se encuentra junto al notebook.

In [ ]:
import pandas as pd
import numpy as np

# TODO: cargar clientes_ecommerce.csv en df
df = pd.read_csv("clientes_ecommerce.csv")

# TODO: primeras cinco filas, shape y dtypes
display(df.head())
print("Shape:", df.shape)
print(df.dtypes)

**Responde:** ¿Cuántas muestras y columnas hay? ¿Qué variable queremos predecir? ¿Es clasificación o regresión?

**Respuesta:** 400 muestras y 12 columnas. Queremos predecir `compra_proximo_mes`. Es clasificación (binaria: Sí/No).

### Ejercicio 2. Separar características y objetivo
Crea `X` excluyendo `cliente_id` y `compra_proximo_mes`. Crea `y` codificando `No` → 0 y `Sí` → 1. Comprueba dimensiones, valores ausentes en `y` y distribución de clases.

In [ ]:
# TODO: crear X sin identificador ni variable objetivo
X = df.drop(columns=["cliente_id", "compra_proximo_mes"])

# TODO: crear y con map({"No": 0, "Sí": 1})
y = df["compra_proximo_mes"].map({"No": 0, "Sí": 1})

# TODO: comprobar X.shape, y.shape, y.isna().sum() y y.value_counts()
print("X:", X.shape)
print("y:", y.shape)
print("Ausentes en y:", y.isna().sum())
print(y.value_counts())

**Responde:** ¿Cuántas características hay? ¿Por qué debemos excluir el identificador y el target de `X`?

**Respuesta:** 10 características. El identificador no dice nada del cliente y el modelo podría aprender patrones falsos; el target hay que quitarlo porque es lo que queremos predecir, si estuviera en `X` el modelo tendría la respuesta.

## Parte 2 - División de datos
### Ejercicio 3. Reservar test
Reserva el 20 % para test mediante `train_test_split` con `random_state=42` y `stratify=y`. El 80 % restante se llamará `X_temp`, `y_temp`.

In [ ]:
from sklearn.model_selection import train_test_split

# TODO: primera división
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# TODO: mostrar tamaños de los dos conjuntos
print("Temp:", X_temp.shape)
print("Test:", X_test.shape)

**Responde:** ¿Para qué sirven `random_state` y `stratify`?

**Respuesta:** `random_state` fija la semilla para que la división salga siempre igual y los resultados se puedan repetir. `stratify` hace que cada conjunto tenga el mismo porcentaje de 0 y 1 que el original.

### Ejercicio 4. Obtener validación
Divide `X_temp`, `y_temp` para que el reparto final sea **60 % train / 20 % validation / 20 % test**. Utiliza `random_state=42` y estratificación.

In [ ]:
# TODO: segunda división
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

**Responde:** ¿Qué valor de `test_size` necesitas en la segunda división? Demuéstralo con una operación.

**Respuesta:** `test_size=0.25`, porque queremos un 20 % del total sacado de un 80 %: 0.20 / 0.80 = 0.25 (0.25 · 320 = 80 muestras).

### Ejercicio 5. Comprobar dimensiones
Muestra `shape` de los seis conjuntos. Verifica que la suma de muestras coincide con el tamaño original. Completa: Train **240** muestras; Validation **80**; Test **80**.

In [ ]:
# TODO: imprimir shape de X_train, X_val, X_test, y_train, y_val, y_test
print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_val:  ", X_val.shape, "| y_val:  ", y_val.shape)
print("X_test: ", X_test.shape, "| y_test: ", y_test.shape)

# TODO: comprobar la suma de tamaños
total = len(X_train) + len(X_val) + len(X_test)
print("Suma:", total, "| Original:", len(df), "| Coinciden:", total == len(df))

## Parte 3 - Verificar los conjuntos
### Ejercicio 6. Distribución de clases
Calcula el porcentaje de las clases 0 y 1 en train, validation y test con `value_counts(normalize=True)`. Crea una tabla comparativa.

In [ ]:
# TODO: proporciones de clases de y_train, y_val y y_test
prop_train = y_train.value_counts(normalize=True)
prop_val = y_val.value_counts(normalize=True)
prop_test = y_test.value_counts(normalize=True)

# TODO: tabla con columnas Train, Validation, Test
(pd.DataFrame({"Train": prop_train, "Validation": prop_val, "Test": prop_test}) * 100).round(2)

**Responde:** ¿Se conservan aproximadamente las proporciones? ¿Por qué importa cuando una clase es minoritaria?

**Respuesta:** Sí, en los tres conjuntos hay ≈ 61-62 % de 0 y ≈ 38-39 % de 1. Importa porque si la clase minoritaria quedara casi sin ejemplos en algún conjunto, el modelo no aprendería bien a detectarla o la evaluación saldría engañosa.

### Ejercicio 7. Independencia de conjuntos
Comprueba con intersecciones de índices que ninguna fila original está presente en dos conjuntos. Usa `assert`. Recuerda que esto **no** detecta automáticamente duplicados con índices distintos.

In [ ]:
# TODO: intersecciones de índices train/val, train/test y val/test
train_val = X_train.index.intersection(X_val.index)
train_test = X_train.index.intersection(X_test.index)
val_test = X_val.index.intersection(X_test.index)
print("Train ∩ Val:", len(train_val))
print("Train ∩ Test:", len(train_test))
print("Val ∩ Test:", len(val_test))

# TODO: tres assert para comprobar que están vacías
assert len(train_val) == 0
assert len(train_test) == 0
assert len(val_test) == 0
print("Conjuntos independientes")

**Responde:** ¿Qué problema habría si se compartieran muestras? ¿Qué limitación tiene esta comprobación?

**Respuesta:** El modelo se evaluaría con datos que ya ha visto y la puntuación saldría mejor de lo real (data leakage). La limitación es que solo mira los índices: si hay filas duplicadas con distinto índice, no las detecta.

## Parte 4 - Data leakage y preprocesamiento
### Ejercicio 8. Detectar errores
Analiza estos casos **sin ejecutarlos**:

**A:** `scaler.fit_transform(X)` antes de dividir en train/test.

**B:** `imputer.fit_transform(X_train)` y `imputer.transform(X_val)`.

**C:** evaluar repetidamente en `X_test` y elegir los hiperparámetros con esa puntuación.

Para cada caso explica si es correcto, cuál es el riesgo y cómo lo corregirías.

**Caso A:** Incorrecto. El escalador calcula media y desviación con todos los datos, incluido test, y eso es data leakage. Se corrige dividiendo primero y haciendo `fit` solo con train (`scaler.fit_transform(X_train)` y `scaler.transform(X_test)`).

**Caso B:** Correcto. El imputador aprende solo de train y a validation solo se le aplica `transform`, así que no hay fuga.

**Caso C:** Incorrecto. Al elegir hiperparámetros mirando test, el test deja de ser independiente y la puntuación final sale optimista. Hay que ajustar con validation (o validación cruzada) y usar test una sola vez al final.

### Ejercicio 9. Edad anómala
Para este ejercicio, edades superiores a 120 se consideran inválidas. Sustitúyelas por `NaN` en **copias** de train, validation y test. Muestra el número de ausentes en `edad` de cada conjunto. No calcules la mediana: lo hará el pipeline.

In [ ]:
X_train = X_train.copy()
X_val = X_val.copy()
X_test = X_test.copy()

# TODO: aplicar regla de edad > 120 en los tres conjuntos
for conjunto in (X_train, X_val, X_test):
    conjunto["edad"] = conjunto["edad"].mask(conjunto["edad"] > 120)

# TODO: mostrar NaN en edad para train/val/test
print("NaN en edad (train):", X_train["edad"].isna().sum())
print("NaN en edad (val):", X_val["edad"].isna().sum())
print("NaN en edad (test):", X_test["edad"].isna().sum())

### Ejercicio 10. Preprocesamiento
Construye tres pipelines y combínalos en `preprocesador`:

- **Numéricas:** imputación mediana + `StandardScaler`.
- **Nominales:** imputación moda + `OneHotEncoder(handle_unknown="ignore", sparse_output=False)`.
- **Ordinal:** imputación moda + `OrdinalEncoder`, orden Bronce, Plata, Oro, Platino y desconocidos `-1`.

**No ajustes el preprocesador por separado con validation ni test.**

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

numericas = [
    "edad", "antiguedad_meses", "ingresos_anuales", "gasto_mensual",
    "num_compras", "devoluciones", "satisfaccion"
]
nominales = ["dispositivo", "zona"]
ordinales = ["nivel_fidelidad"]

# TODO: pipeline_numerico = Pipeline([...])
pipeline_numerico = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# TODO: pipeline_nominal = Pipeline([...])
pipeline_nominal = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# TODO: pipeline_ordinal = Pipeline([...])
pipeline_ordinal = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        categories=[["Bronce", "Plata", "Oro", "Platino"]],
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

# TODO: preprocesador = ColumnTransformer([...])
preprocesador = ColumnTransformer([
    ("num", pipeline_numerico, numericas),
    ("nom", pipeline_nominal, nominales),
    ("ord", pipeline_ordinal, ordinales)
])

**Responde:** ¿Qué ventajas tiene mantener todas las transformaciones en un único preprocesador?

**Respuesta:** Se ajusta todo de una vez solo con train y se aplica igual a validation y test, sin riesgo de fugas ni de olvidar algún paso. Además el código queda más limpio y fácil de reutilizar.

## Parte 5 - Primer modelo
### Ejercicio 11. Pipeline completo
Construye `modelo` con dos etapas: `preprocesamiento` (el objeto anterior) y `clasificador` (`LogisticRegression(max_iter=1000)`).

In [ ]:
from sklearn.linear_model import LogisticRegression

# TODO: modelo = Pipeline([...])
modelo = Pipeline([
    ("preprocesamiento", preprocesador),
    ("clasificador", LogisticRegression(max_iter=1000))
])

**Responde:** ¿Qué ocurre antes de entrenar el clasificador? ¿Cómo se tratará una categoría nominal nueva?

**Respuesta:** Antes de entrenar se imputan los ausentes, se escalan las numéricas y se codifican las categóricas. Una categoría nominal nueva no da error: con `handle_unknown="ignore"` queda con todo ceros en sus columnas.

### Ejercicio 12. Entrenar
Entrena el pipeline **solo** con `X_train` e `y_train`.

In [ ]:
# TODO: modelo.fit(...)
modelo.fit(X_train, y_train)

**Responde:** ¿Qué aprende el preprocesador? ¿Qué aprende el clasificador?

**Respuesta:** El preprocesador aprende medianas y modas para imputar, media y desviación para escalar y las categorías de cada columna. El clasificador aprende los coeficientes de la regresión logística (el peso de cada característica).

### Ejercicio 13. Predecir
Crea `predicciones_train` y `predicciones_val` usando `modelo.predict()`. Muestra las primeras diez predicciones de validación. **No uses `X_test`**.

In [ ]:
# TODO: predicciones_train = ...
predicciones_train = modelo.predict(X_train)

# TODO: predicciones_val = ...
predicciones_val = modelo.predict(X_val)

# TODO: primeras diez predicciones de validación
print(predicciones_val[:10])

**Responde:** ¿Qué significan 0 y 1? ¿Por qué no hay que transformar manualmente `X_val` antes de `predict()`?

**Respuesta:** 0 significa que el cliente no comprará el próximo mes y 1 que sí. No hace falta transformar `X_val` porque el pipeline ya le aplica el preprocesador (con lo aprendido en train) antes de predecir.

### Ejercicio 14. Accuracy
Calcula `accuracy_train` y `accuracy_val`. Muestra una tabla con ambos resultados. No se permite calcular accuracy sobre test.

In [ ]:
from sklearn.metrics import accuracy_score

# TODO: accuracy_train = ...
accuracy_train = accuracy_score(y_train, predicciones_train)

# TODO: accuracy_val = ...
accuracy_val = accuracy_score(y_val, predicciones_val)

# TODO: tabla comparativa
pd.DataFrame({
    "Conjunto": ["Train", "Validation"],
    "Accuracy": [accuracy_train, accuracy_val]
}).round(3)

**Responde:** ¿Qué conjunto obtiene mayor accuracy? ¿Hay diferencia importante? ¿Una accuracy alta garantiza que el modelo sea bueno?

**Respuesta:** Train, con 0.708 frente a 0.663 en validation. Hay unos 4.5 puntos de diferencia, no es muy grande. No, una accuracy alta no garantiza nada: con clases desbalanceadas (≈ 62 % de 0) un modelo que diga siempre 0 ya saca ≈ 0.62, hay que mirar también otras métricas.

### Ejercicio 15. Comparar predicciones
Crea un DataFrame `resultados` con las columnas `Real`, `Predicción` y `Correcta`. Muestra las primeras 15 filas y cuenta aciertos y errores.

In [ ]:
# TODO: resultados = pd.DataFrame({...})
resultados = pd.DataFrame({
    "Real": y_val.values,
    "Predicción": predicciones_val,
    "Correcta": y_val.values == predicciones_val
}, index=y_val.index)

# TODO: primeras 15 filas
display(resultados.head(15))

# TODO: total de predicciones correctas e incorrectas
print("Correctas:", resultados["Correcta"].sum())
print("Incorrectas:", (~resultados["Correcta"]).sum())

**Responde:** ¿Qué información adicional necesitaríamos para analizar mejor los errores del clasificador?

**Respuesta:** Saber qué tipo de error comete: la matriz de confusión (falsos positivos y falsos negativos), y métricas como precision y recall. También ver las características de los clientes mal clasificados.

## Parte 6 - Interpretación y conclusiones
### Ejercicio 16. Analizar resultados hipotéticos
| Modelo | Accuracy train | Accuracy validation |
|---|---:|---:|
| A | 0.99 | 0.61 |
| B | 0.83 | 0.81 |
| C | 0.55 | 0.53 |

Responde: ¿Cuál muestra indicios de sobreajuste? ¿Cuál es más consistente? ¿Cuál podría estar infraajustado? ¿Elegirías el de mayor accuracy de entrenamiento? ¿Qué comprobarías antes de decidir?

**Respuesta:**

- **Sobreajuste:** A (0.99 en train y 0.61 en validation).
- **Más consistente:** B, casi no hay diferencia entre train y validation.
- **Infraajuste:** C, va mal en los dos.
- No, el de mayor accuracy de train es A y es el que peor generaliza. Elegiría B.
- Antes de decidir miraría la distribución de clases, la matriz de confusión y otras métricas (precision, recall), y al final lo evaluaría una sola vez en test.